# MFLib Meas Node — Slice Creation

Creates a FABRIC slice with a single meas node connected via FABNetv6 (`type="IPv6"`).

## What this notebook does

| Step | Cell | Description |
|------|------|-------------|
| 1 | 1 | Imports & configuration |
| 2 | 2 | Initialize fablib |
| 3 | 3 | Define slice topology |
| 4 | 4 | Submit slice and wait for StableOK |
| 5 | 5 | Collect node info |
| 6 | 6 | Assign static FABNetv6 IP (FABRIC ACL requires this) |
| 7 | 7 | Install persistent policy routing (table 30, systemd) |
| 8 | 8 | Create mfuser account and SSH key pair |
| 9 | 9 | Deploy FastAPI info/registration server on port 5000 |
| 10 | 10 | Write local slice info to `/etc/mflib/portal_registration.json` |
| 11 | 11 | Test portal connectivity (reachability check) |
| 12 | 12 | **Register with portal** *(separate cell — run independently)* |
| 13 | 13 | Install MeasurementFramework |
| 14 | 14 | Summary |

## Server source files

The FastAPI server lives in `meas-node-server/` at the repo root.  Cell 9 uploads
all files and installs them under `/etc/mflib/server/` on the meas node.

| Deployed path | Purpose |
|---------------|---------|
| `/etc/mflib/server/main.py` | FastAPI app entry point (port 5000, binds `::`) |
| `/etc/mflib/server/schemas.py` | Pydantic models for registration payload |
| `/etc/mflib/server/storage.py` | Multi-slice JSON store + service allowlist |
| `/etc/mflib/server/routers/info.py` | `/status`, `/ip`, `/slice`, `/registration` |
| `/etc/mflib/server/routers/register.py` | `POST /register`, `GET/DELETE /registrations/{id}` |
| `/etc/mflib/server/routers/mflib_ops.py` | `/mflib/{slice_id}/...` mflib operations |
| `/etc/systemd/system/mflib-info-server.service` | systemd unit (User=mfuser) |

The routing script and its systemd unit are **generated in Cell 7** with the actual IP addresses and device names discovered after slice submission.

In [ ]:
from datetime import datetime

now = datetime.now()
print(now)                                    # 2026-07-04 14:32:07.123456
print(f'This notebook was run {now.strftime("%A, %B %d, %Y at %I:%M %p")}')  # Saturday, July 04, 2026 at 02:32 PM

## Cell 1 — Imports & Configuration

In [ ]:
%%time
import base64, io, json
from datetime import datetime, timezone
from pathlib import Path

import paramiko


In [ ]:
%%time
def get_unique_slice_name(fablib, name: str) -> str:
    """
    Returns the given slice name if it doesn't exist, otherwise increments
    the last character to the next capital letter (A→B, B→C, ... Z→A).
    
    :param fablib: FablibManager instance
    :param name: desired slice name
    :return: unique slice name
    """
    existing = {s.get_name() for s in fablib.get_slices()}
    
    if name not in existing:
        return name
    
    base = name[:-1] if name and name[-1].isupper() else name
    suffix_ord = ord(name[-1]) if name and name[-1].isupper() else ord('A') - 1
    
    while True:
        next_char = chr((suffix_ord - ord('A') + 1) % 26 + ord('A'))
        candidate = base + next_char
        if candidate not in existing:
            return candidate
        suffix_ord = ord(next_char)

## Debug Methods

## Cell 2 — Initialize fablib

In [ ]:
%%time
from fabrictestbed_extensions.fablib.fablib import FablibManager 


try:
    fm = FablibManager()
    fm.show_config()
except Exception as e:
    print(f'ERROR: {e}')
    raise

## Cell 3 — Define Slice Topology

One node + one FABNetv6 (`type="IPv6"`) network.  FABNetv6 provides a FABRIC-internal /64 IPv6 subnet reachable only within the same project's slices.

In [ ]:
%%time
SLICE_NAME = get_unique_slice_name(fm, "MeasNode_A")
print(f"Using slice {SLICE_NAME}")

In [ ]:
%%time
# ── User-editable ─────────────────────────────────────────────────────────
#SLICE_NAME        = "MeasNode01"
SITE              = "EDC"          # e.g. "TACC"; None = auto-select
IMAGE             = "docker_ubuntu_20"
CORES             = 4
RAM_GB            = 16
DISK_GB           = 100

MEAS_NODE_NAME    = "meas-node"
MEAS_NETWORK_NAME = "meas-net6"
MEAS_NIC_NAME     = "meas-nic6"

# Optional: reuse existing keys across slices.  Leave None to generate new.
MFUSER_KEY_PATH   = None   # e.g. 'MeasNode01_mfuser.key'
MFUSER_PUB_PATH   = None   # e.g. 'MeasNode01_mfuser.pub'

############################################################################################################################################################################
################################################################################# SET PORTAL IP ############################################################################
############################################################################################################################################################################
# Set to the portal's public IPv4 URL to enable Cell 11 (registration).
PORTAL_PUBLIC_URL = 'http://23.134.232.150'  #None   # e.g. 'http://23.134.232.146'

# iproute2 policy routing table for FABNetv6 (must not clash with 0/253/254/255)
RT_V6 = 30

# Directory containing node files to upload
NODE_FILES_DIR = Path('.')

print('Configuration loaded.')
print(f'  Slice : {SLICE_NAME}  @  {SITE or "auto-select"}')
print(f'  Portal: {PORTAL_PUBLIC_URL or "(not set — registration will be skipped in Cell 11)"}')

In [ ]:
%%time
slice_obj = fm.new_slice(name=SLICE_NAME)

meas_node = slice_obj.add_node(name=MEAS_NODE_NAME, site=SITE)
meas_node.set_capacities(cores=CORES, ram=RAM_GB, disk=DISK_GB)
meas_node.set_image(IMAGE)

iface = meas_node.add_component(model='NIC_Basic', name=MEAS_NIC_NAME).get_interfaces()[0]
meas_net = slice_obj.add_l3network(name=MEAS_NETWORK_NAME, interfaces=[iface], type='IPv6')

print('Topology defined:')
print(f'  {MEAS_NODE_NAME} <-- {MEAS_NIC_NAME} --> {MEAS_NETWORK_NAME}  (FABNetv6)')

## Cell 4 — Submit Slice

In [ ]:
%%time
print(f"Submitting '{SLICE_NAME}' (5-10 min)…")
slice_obj.submit(wait=True, wait_timeout=600, wait_interval=20, progress=True)
print(f'\nSlice up — ID: {slice_obj.get_slice_id()}')

## Cell 5 — Collect Node Info

In [ ]:
%%time
node          = slice_obj.get_node(MEAS_NODE_NAME)
node_mgmt_ip  = str(node.get_management_ip()) if node.get_management_ip() else None
node_ssh_cmd  = node.get_ssh_command()
node_username = node.get_username()
slice_id      = slice_obj.get_slice_id()

print(f'Slice ID : {slice_id}')
print(f'Mgmt IP  : {node_mgmt_ip}')
print(f'SSH      : {node_ssh_cmd}')
print(f'Username : {node_username}')

## Cell 6 — Assign Static FABNetv6 IP

FABRIC's ACL only allows cross-slice traffic to/from the **registered static address**, not the SLAAC/EUI-64 address assigned by `post_boot_config()`.

`net_obj.config()` may assign the address at the infrastructure level; the cell checks first to avoid a redundant (and permission-denied) `ip addr add`.

In [ ]:
%%time
net_obj   = slice_obj.get_network(MEAS_NETWORK_NAME)
iface_obj = node.get_interface(network_name=MEAS_NETWORK_NAME)
net_obj.config()

subnet_v6 = net_obj.get_subnet()
gw_v6     = net_obj.get_gateway()
node_ipv6 = net_obj.get_available_ips(count=1)[0]
dev       = iface_obj.get_device_name()

print(f'Subnet  : {subnet_v6}')
print(f'Gateway : {gw_v6}')
print(f'IP      : {node_ipv6}')
print(f'Device  : {dev}')

# Only call ip_addr_add() if the address is not already present
_check, _ = node.execute(f'ip -6 addr show dev {dev}')
if str(node_ipv6) not in _check:
    node.ip_addr_add(addr=node_ipv6, subnet=subnet_v6, interface=iface_obj)
    print('Static IPv6 assigned.')
else:
    print('Address already present — skipping ip_addr_add().')

node_ipv6       = str(node_ipv6)
meas_net_subnet = str(subnet_v6)
gw_v6_str       = str(gw_v6)

stdout, _ = node.execute(f'ip -6 addr show {dev}')
print(stdout)

## Cell 7 — Persistent FABNetv6 Routing

Installs policy routing table 30 so replies from `node_ipv6` exit via the FABNetv6 gateway rather than the management default.

The routing script is **generated here** with the actual IP / device values discovered in Cell 6, written to a local temp file, uploaded with `node.upload_file()`, and enabled as a systemd one-shot service so it survives reboots.

In [ ]:
%%time
routing_script = '\n'.join([
    '#!/usr/bin/env bash',
    '# /etc/mflib/fabnetv6_routing.sh — MFLib meas-node FABNetv6 policy routing',
    '# Generated by create-meas-node.ipynb — idempotent, safe to re-run.',
    'set -euo pipefail',
    '',
    f'NODE_IPV6="{node_ipv6}"',
    f'SUBNET_V6="{meas_net_subnet}"',
    f'GW_V6="{gw_v6_str}"',
    f'DEV="{dev}"',
    f'RT_V6={RT_V6}',
    '',
    'ip -6 route flush table $RT_V6 2>/dev/null || true',
    'ip -6 route add $SUBNET_V6 dev $DEV scope link table $RT_V6',
    'ip -6 route add default via $GW_V6 dev $DEV table $RT_V6',
    'ip -6 rule show | grep -qF "from $NODE_IPV6 lookup $RT_V6" || \\',
    '    ip -6 rule add from $NODE_IPV6 table $RT_V6 priority 300',
    'echo "[mflib] FABNetv6 routing table $RT_V6 applied"',
]) + '\n'

routing_unit = '\n'.join([
    '[Unit]',
    f'Description=MFLib meas-node FABNetv6 policy routing (table {RT_V6})',
    'After=network-online.target',
    'Wants=network-online.target',
    '',
    '[Service]',
    'Type=oneshot',
    'ExecStart=/etc/mflib/fabnetv6_routing.sh',
    'RemainAfterExit=yes',
    '',
    '[Install]',
    'WantedBy=multi-user.target',
]) + '\n'

with open('/tmp/fabnetv6_routing.sh', 'w') as f:
    f.write(routing_script)
with open('/tmp/mflib-fabnetv6.service', 'w') as f:
    f.write(routing_unit)

node.upload_file('/tmp/fabnetv6_routing.sh',    '/tmp/fabnetv6_routing.sh')
node.upload_file('/tmp/mflib-fabnetv6.service', '/tmp/mflib-fabnetv6.service')

stdout, _ = node.execute(
    'sudo mkdir -p /etc/mflib && '
    'sudo cp /tmp/fabnetv6_routing.sh /etc/mflib/fabnetv6_routing.sh && '
    'sudo chmod +x /etc/mflib/fabnetv6_routing.sh && '
    'sudo cp /tmp/mflib-fabnetv6.service /etc/systemd/system/mflib-fabnetv6.service && '
    'sudo systemctl daemon-reload && '
    'sudo systemctl enable mflib-fabnetv6.service && '
    'sudo /etc/mflib/fabnetv6_routing.sh'
)
print(stdout)

## Cell 8 — mfuser Account Setup

In [ ]:
%%time
# ── Key pair ────────────────────────────────────────────────────────────
if MFUSER_KEY_PATH and MFUSER_PUB_PATH:
    print(f'Loading mfuser keys from {MFUSER_KEY_PATH}')
    with open(MFUSER_KEY_PATH) as f:
        mfuser_private_key = f.read()
    with open(MFUSER_PUB_PATH) as f:
        mfuser_public_key = f.read().strip()
else:
    save_prefix = f'{SLICE_NAME}_mfuser'
    key = paramiko.RSAKey.generate(2048)
    buf = io.StringIO()
    key.write_private_key(buf)
    mfuser_private_key = buf.getvalue()
    mfuser_public_key  = f'ssh-rsa {key.get_base64()} mfuser'
    with open(f'{save_prefix}.key', 'w') as f:
        f.write(mfuser_private_key)
    with open(f'{save_prefix}.pub', 'w') as f:
        f.write(mfuser_public_key + '\n')
    print(f'Keys saved: {save_prefix}.key  /  {save_prefix}.pub')

# ── Create account on node ───────────────────────────────────────────────
cmd = ' && '.join([
    'sudo useradd -s /bin/bash -G root -m mfuser || true',
    'sudo mkdir -p /home/mfuser/.ssh',
    'sudo chmod 700 /home/mfuser/.ssh',
    "echo 'mfuser ALL=(ALL:ALL) NOPASSWD: ALL' | sudo tee /etc/sudoers.d/mfuser",
    f"echo '{mfuser_public_key}' | sudo tee /home/mfuser/.ssh/authorized_keys",
    'sudo chmod 644 /home/mfuser/.ssh/authorized_keys',
    'sudo chown -R mfuser:mfuser /home/mfuser/.ssh',
])
stdout, stderr = node.execute(cmd)
print('mfuser account ready.')
if stderr:
    print('stderr:', stderr[:200])

## Cell 9 — Deploy FastAPI Info/Registration Server

Uploads the FastAPI server from `meas-node-server/` (repo root) to `/etc/mflib/server/`
on the meas node, installs pip dependencies, and starts the systemd service.

| Route | Method | Returns |
|-------|--------|---------|
| `/status` | GET | uptime, hostname, timestamp |
| `/ip` | GET | `ip -j addr show` output |
| `/slice` | GET | `portal_registration.json` |
| `/registration` | GET | all registered slices |
| `/register` | POST | store a slice registration |
| `/registrations/{slice_id}` | GET/DELETE | per-slice record |
| `/mflib/{slice_id}/status` | GET | `mflib.check()` |
| `/mflib/{slice_id}/instrumentize` | POST | install measurement tools |
| `/mflib/{slice_id}/start/{service}` | POST | start a service |
| `/mflib/{slice_id}/stop/{service}` | POST | stop a service |
| `/mflib/{slice_id}/download` | GET | retrieve collected data |
| `/mflib/allowed-services` | GET | configured service allowlist |

The portal queries `/status`, `/ip`, and `/slice` via `GET /api/meas-node/{slice_uuid}/info`.

In [ ]:
%%time
# meas-node-server/ is at the repo root; notebook is in Notebooks/meas-node-creation/
SERVER_DIR = Path(__file__).parent.parent.parent / 'meas-node-server' \
    if '__file__' in dir() else Path('../../meas-node-server')

# Verify the source directory is present
if not SERVER_DIR.exists():
    raise FileNotFoundError(
        f"Server source not found at {SERVER_DIR.resolve()}. "
        "Run this notebook from its own directory or adjust SERVER_DIR."
    )
print(f"Server source: {SERVER_DIR.resolve()}")

# ── Create remote directory layout ───────────────────────────────────────
node.execute('sudo mkdir -p /etc/mflib/server/routers')
node.execute('sudo chown -R mfuser:mfuser /etc/mflib/server')

# ── Upload top-level server files ─────────────────────────────────────────
top_level_files = ['main.py', 'schemas.py', 'storage.py', 'requirements.txt']
for fname in top_level_files:
    src = SERVER_DIR / fname
    node.upload_file(str(src), f'/tmp/mfserver_{fname}')
    node.execute(f'sudo cp /tmp/mfserver_{fname} /etc/mflib/server/{fname}')
    print(f"  uploaded {fname}")

# ── Upload router files ────────────────────────────────────────────────────
router_files = ['__init__.py', 'info.py', 'register.py', 'mflib_ops.py']
for fname in router_files:
    src = SERVER_DIR / 'routers' / fname
    if not src.exists():
        print(f"  skipping routers/{fname} (not found)")
        continue
    node.upload_file(str(src), f'/tmp/mfrouter_{fname}')
    node.execute(f'sudo cp /tmp/mfrouter_{fname} /etc/mflib/server/routers/{fname}')
    print(f"  uploaded routers/{fname}")

# ── Install Python dependencies ────────────────────────────────────────────
print("\nInstalling pip dependencies…")
stdout, stderr = node.execute(
    'sudo pip3 install -q -r /etc/mflib/server/requirements.txt'
)
if stdout:
    print(stdout)
if stderr and 'WARNING' not in stderr:
    print("pip stderr:", stderr[:400])

# ── Install and start systemd service ─────────────────────────────────────
node.upload_file(
    str(SERVER_DIR / 'mflib-info-server.service'),
    '/tmp/mflib-info-server.service',
)
stdout, _ = node.execute(
    'sudo cp /tmp/mflib-info-server.service /etc/systemd/system/mflib-info-server.service && '
    'sudo systemctl daemon-reload && '
    'sudo systemctl enable mflib-info-server.service && '
    'sudo systemctl restart mflib-info-server.service'
)
print(stdout)

# ── Verify ────────────────────────────────────────────────────────────────
import time as _time
_time.sleep(3)
stdout, _ = node.execute(
    'sudo systemctl is-active mflib-info-server.service && '
    'curl -sf http://[::1]:5000/status || echo "WARNING: /status not yet responding"'
)
print(stdout)
print(f'\nFastAPI server started — http://[{node_ipv6}]:5000/status')

# Cell 10 — Write Slice Info File

Writes local slice info to `/etc/mflib/portal_registration.json` on the meas node. This file is served by `GET /slice` on the info server.

Written unconditionally so the info server always has something useful to return, even when no portal is configured.  Cell 11 overwrites it with full registration data after a successful portal call.

In [ ]:
%%time
def _write_json_to_node(node_obj, data, remote_path):
    """Write data as JSON to remote_path using a base64 pipe (avoids quoting issues)."""
    encoded = base64.b64encode(json.dumps(data, indent=2).encode()).decode()
    stdout, stderr = node_obj.execute(
        f'echo {encoded} | base64 -d | sudo tee {remote_path} > /dev/null'
    )

local_info = {
    'written_at':       datetime.now(timezone.utc).isoformat(),
    'slice_id':         slice_id,
    'slice_name':       slice_obj.get_name(),
    'node_mgmt_ip':     node_mgmt_ip,
    'node_ipv6':        node_ipv6,
    'meas_net_subnet':  meas_net_subnet,
    'meas_net_gateway': gw_v6_str,
    'lease_start':      str(slice_obj.get_lease_start()),
    'lease_end':        str(slice_obj.get_lease_end()),
    'portal_registration': None,
}
print(local_info)
_write_json_to_node(node, local_info, '/etc/mflib/portal_registration.json')
print('Slice info written to /etc/mflib/portal_registration.json')

## Cell 11 — Test Portal Connectivity

Verifies the portal is reachable and its FABNetv6 info endpoint responds before attempting registration.  Safe to re-run at any time.

- `PORTAL_PUBLIC_URL` not set → skips with a reminder.
- Portal reachable → prints portal FABNetv6 details.
- Portal unreachable → raises `RuntimeError` so Cell 12 is never run against a dead URL.

In [ ]:
%%time
import requests as _req

if not PORTAL_PUBLIC_URL:
    print('PORTAL_PUBLIC_URL is not set.')
    print('Set it in Cell 1 and re-run Cells 1, 11, and 12.')
else:
    _portal_url   = PORTAL_PUBLIC_URL.rstrip('/')
    _health_url   = f'{_portal_url}/api/meas-node/portal-info'
    print(f'Checking portal at {_health_url} …')
    try:
        _r = _req.get(_health_url, timeout=10)
        _r.raise_for_status()
        _info = _r.json()
        print('Portal reachable.')
        print(f"  FABNetv6 IP      : {_info.get('fabnetv6_ip')}")
        print(f"  FABNetv6 subnet  : {_info.get('fabnetv6_subnet')}")
        print(f"  FABNetv6 gateway : {_info.get('fabnetv6_gateway')}")
        print('\nReady to register — proceed to Cell 12.')
    except Exception as _exc:
        raise RuntimeError(
            f'Portal unreachable at {_health_url}: {_exc}\n'
            'Fix PORTAL_PUBLIC_URL in Cell 3, re-run Cell 3, then re-run this cell.'
        ) from None


## Cell 12 — Register with Portal

Calls `POST /api/meas-node/register` on the MFLib portal to:
- Tell the portal this node's FABNetv6 address and subnet
- Get back the portal's own FABNetv6 info (needed to configure the return route)

After registration the portal will:
- Start SSH-probing this node every 60 s (15 min), then every 15 min
- The `route-watcher` sidecar adds `ip -6 route` for this node's subnet within 30 s

**Prerequisite:** set `PORTAL_PUBLIC_URL` in Cell 1 and re-run that cell first.

In [ ]:
%%time
import requests as _req

portal_registration = None

if not PORTAL_PUBLIC_URL:
    print('PORTAL_PUBLIC_URL is not set — skipping registration.')
    print('Set it in Cell 1 and re-run this cell to register.')
else:
    portal_url = PORTAL_PUBLIC_URL.rstrip('/')
    reg_url    = f'{portal_url}/api/meas-node/register'

    reg_payload = {
        'slice_uuid':        slice_id,
        'slice_name':        SLICE_NAME,
        'fabnetv6_ip':       node_ipv6,
        'fabnetv6_subnet':   meas_net_subnet,
        'fabnetv6_gateway':  gw_v6_str,
        'mfuser_public_key': mfuser_public_key,
        'node_mgmt_ip':      node_mgmt_ip,
        'slice_created_at':  str(slice_obj.get_lease_start()),
        'slice_expires_at':  str(slice_obj.get_lease_end()),
    }

    print(f'Registering with portal at {reg_url} …')
    try:
        resp = _req.post(reg_url, json=reg_payload, timeout=30)
        resp.raise_for_status()
        reg_response = resp.json()
        print(f"Status   : {reg_response.get('status')}")
        print(json.dumps(reg_response, indent=2))
    except Exception as exc:
        print(f'Registration failed: {exc}')
        reg_response = {'error': str(exc)}

    portal_registration = {
        'registered_at': datetime.now(timezone.utc).isoformat(),
        'portal_url':    portal_url,
        'request':       reg_payload,
        'response':      reg_response,
    }

    # Overwrite the slice info file with full registration data
    local_info['portal_registration'] = portal_registration
    local_info['written_at'] = datetime.now(timezone.utc).isoformat()
    _write_json_to_node(node, local_info, '/etc/mflib/portal_registration.json')
    print('\nRegistration record saved to /etc/mflib/portal_registration.json')

## Cell 13 — Install MeasurementFramework

In [ ]:
%%time

# TODO change to downloading a release tarball instead of cloning the repo

mf_repo_branch = "main"
cmd = f"sudo -u mfuser git clone -q -b {mf_repo_branch} https://github.com/fabric-testbed/MeasurementFramework.git /home/mfuser/mf_git"
stdout, stderr = node.execute(cmd, quiet=True)

if stdout:
    print(f"STDOUT: {stdout}")
if stderr:
    if "already exists and is not an empty directory" not in stderr:
        msg = (
            f"Clone Directory already exist. Cloning Measurement Framework Repository from github.com Failed."
        )
        print(msg)
    else:
        print(f"STDERR: {stderr}")

In [ ]:
%%time
 #def _run_bootstrap_script(self):
"""
Run the initial bootstrap script in the meas node mf repo.
"""
msg = f"Starting Bootstrap Process on Measure Node (bash script)..."
#self.core_logger.debug(msg)
print(msg)

cmd = f"sudo -u mfuser /home/mfuser/mf_git/instrumentize/experiment_bootstrap/bootstrap.sh"

stdout, stderr = node.execute(cmd, quiet=True)

msg = f"Bootstrap Process on Measure Node (bash script) done."
#self.core_logger.debug(msg)
print(msg)

if stdout:
    print(f"STDOUT: {stdout}")
if stderr:
    print(f"STDERR: {stderr}")

In [ ]:
%%time
#def _run_bootstrap_ansible(self):
"""
Run the initial bootstrap ansible scripts in the meas node mf repo.
"""
msg = f"Starting Bootstrap Process on Measure Node (Ansible Playbook)..."
#self.core_logger.debug(msg)
print(msg)

cmd = (
    f"sudo cp /home/mfuser/mf_git/instrumentize/experiment_bootstrap/ansible.cfg /home/mfuser/services/common/ansible.cfg;"
    f"sudo chown mfuser:mfuser /home/mfuser/services/common/ansible.cfg;"
    f"sudo -u mfuser python3 /home/mfuser/mf_git/instrumentize/experiment_bootstrap/bootstrap_playbooks.py;"
)
stdout, stderr = node.execute(cmd, quiet=True)

msg = f"Bootstrap Process on Measure Node (Ansible Playbook) done."
#self.core_logger.debug(msg)
print(msg)

if stdout:
    try:
        print(f"STDOUT: {json.dumps(stdout, indent=2)}")
    except ValueError as e:
        print(f"STDOUT: {stdout}")
    if "Bootstrap playbook install failed" in stdout:
        print("Bootstrap ansible scripts Failed. See logs for details")
        #return False
if stderr:
    print(f"STDERR: {stderr}")

print("Bootstrap ansible scripts done")
#return True

In [ ]:
%%time
f_repo_branch = "node"
cmd = ( f"sudo -u mfuser git clone -q -b {f_repo_branch} https://github.com/fabric-testbed/mflib.git /home/mfuser/mflib;"
        f"cd /home/mfuser/mflib;"
        f"sudo -u mfuser pip install -e mflib-node;"
      )
stdout, stderr = node.execute(cmd, quiet=True)

if stdout:
    print(f"STDOUT: {stdout}")
if stderr:
    if "already exists and is not an empty directory" not in stderr:
        msg = (
            f"Clone Directory already exist. CloningMFLIB Repository from github.com Failed."
        )
        print(msg)
    else:
        print(f"STDERR: {stderr}")

## Cell 14 — Summary

In [ ]:
%%time
sep = '=' * 62
print(sep)
print(f'  Meas Node Ready — {SLICE_NAME}')
print(sep)
print(f'  Slice ID      : {slice_id}')
print(f'  FABNetv6 IP   : {node_ipv6}')
print(f'  Subnet        : {meas_net_subnet}')
print(f'  Gateway       : {gw_v6_str}')
print(f'  Mgmt IP       : {node_mgmt_ip}')
print(f'  SSH           : {node_ssh_cmd}')
print(f'  mfuser key    : {SLICE_NAME}_mfuser.key')
print('-' * 62)
print(f'  Info server   : http://[{node_ipv6}]:5000/status')
if portal_registration:
    print(f'  Portal        : registered')
    print(f'  Portal info   : {PORTAL_PUBLIC_URL}/api/meas-node/{slice_id}/info')
    # Proxy URL shown if PORTAL_DOMAIN is configured on the portal
    import re as _re
    _slug = _re.sub(r'[^a-z0-9]+', '-', SLICE_NAME.lower()).strip('-') + '-' + slice_id[:5]
    print(f'  Proxy URL     : http://{_slug}.<PORTAL_DOMAIN>/status  (set PORTAL_DOMAIN in docker-compose)')
else:
    print('  Portal        : not registered  (run Cell 11 with PORTAL_PUBLIC_URL set)')
print(sep)

## Cell 15 — (Optional) Delete Slice

In [ ]:
# Uncomment to release all FABRIC resources for this slice:
# slice_obj = fm.get_slice(name=SLICE_NAME)
# slice_obj.delete()
# print(f"Slice '{SLICE_NAME}' deleted.")